# Defining a reasonable decision bondary for the 2D classification problem



In [ ]:
import os
import numpy as np
import pandas as pd 
import glob
from matplotlib import font_manager
from matplotlib import rcParams

import matplotlib.pyplot as plt

def setup_dirs(outDir):
    figuresDir = os.path.join(outDir, "figures")
    tablesDir = os.path.join(outDir, "tables")
    dataDir = os.path.join(outDir, "data")
    os.makedirs(dataDir, exist_ok=True)
    os.makedirs(figuresDir, exist_ok=True)
    os.makedirs(tablesDir, exist_ok=True)
    return figuresDir, tablesDir, dataDir


def force_arial():
    arial_font_path = "/home/salehis/projects/cdm/fonts/arial.ttf"
    font_manager.fontManager.addfont(arial_font_path)
    prop = font_manager.FontProperties(fname=arial_font_path)
    print("Arial font forced")


force_arial()

rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

outDir = '/data1/shahs3/users/salehis/ecdna/results/decision_boundary_nov_29_2025'
figuresDir, tablesDir, dataDir = setup_dirs(outDir)


rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/decision_boundary_nov_29_2025/figures/*.p* \
    /Users/salehis/Projects/ecdna

In [ ]:
def get_process_color(df, proces_col='process'):
    # for each fucking process that fucking exists, assign a separate color
    processes = df[proces_col].unique()
    colors = plt.cm.get_cmap('tab10', len(processes))
    color_dict = {processes[i]: colors(i) for i in range(len(processes))}
    return df[proces_col].map(color_dict)


In [ ]:
data_df__path = '/data1/shahs3/users/leej39/dlp/analysis/modeling/amplified_regions_summary_table_111925.csv'
df = pd.read_csv(data_df__path)
df.columns

# filter by mean_cn > 6
df = df[df['mean_cn'] > 6].copy()

# Plot a scatter plot: X: mass_in_window, Y: score, save to file

plt.clf()
plt.figure(figsize=(6,6))
plt.scatter(df['mass_in_window'], df['score'], alpha=0.5)
plt.xlabel('Mass in window')
plt.ylabel('Score')
plt.title('Scatter plot of Mass in window vs Score')
plt.grid(True)
plt.legend()
output_path = os.path.join(figuresDir, 'mass_vs_score_scatter_plot.png')
plt.savefig(output_path, dpi=300)
plt.close()



In [ ]:
# label all points where the x is betewen .5 and .3 and y is .6 and .8
from adjustText import adjust_text

df_label = df[(df['mass_in_window'] > 0.4) & (df['mass_in_window'] < 0.8) & (df['score'] > 0.3) & (df['score'] < 0.6)]
plt.clf()
plt.figure(figsize=(6,6))
# set color to process column

# Ensure that there is a legend for the process colors
plt.scatter(df['mass_in_window'], df['score'], c=get_process_color(df), alpha=0.5)
plt.xlabel('Mass in window')
plt.ylabel('Score')
plt.title('Scatter plot of Mass in window vs Score with Labels')
plt.grid(True)
texts = []
for i, row in df_label.iterrows():
    # add an arrow from the point to the text
    #_ = texts.append(plt.text(row['mass_in_window'], row['score'], str(row['individual']), fontsize=4, arrowprops=dict(arrowstyle="->", color='r')))
    # it's erroing out, so please add the arrow correctly
    _ = texts.append(plt.text(row['mass_in_window'], row['score'], str(row['individual']), fontsize=4))
    # add an arrow from the point to the text, using adjust text
_ = adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
output_path = os.path.join(figuresDir, 'mass_vs_score_scatter_plot_labeled.png')
# legend
processes = df['process'].unique()
colors = plt.cm.get_cmap('tab10', len(processes))
for i, process in enumerate(processes):
    plt.scatter([], [], c=[colors(i)], label=process)
plt.legend()
plt.savefig(output_path, dpi=300)
plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_boundary_from_quadrants(df):
    x = df['mass_in_window'].values
    y = df['score'].values
    x_med = np.median(x)
    y_med = np.median(y)
    mask_A = (x < x_med) & (y > y_med)   # upper left
    mask_B = (x > x_med) & (y < y_med)   # lower right
    S_A = np.column_stack([x[mask_A], y[mask_A]])
    S_B = np.column_stack([x[mask_B], y[mask_B]])
    if S_A.shape[0] == 0 or S_B.shape[0] == 0:
        raise ValueError("Not enough points in one of the anchor quadrants.")
    mu_A = S_A.mean(axis=0)
    mu_B = S_B.mean(axis=0)
    w = mu_B - mu_A              # normal vector to boundary
    mid = 0.5 * (mu_A + mu_B)    # midpoint of centroids
    return w, mid

def plot_decision_boundary(df, q=0.3, center_weight=0.0, save_path=None):
    w, mid = compute_boundary_from_quadrants(df, q=q, center_weight=center_weight)
    x_vals = df['mass_in_window'].values
    y_vals = df['score'].values
    plt.figure(figsize=(6,6))
    plt.scatter(x_vals, y_vals, s=10, alpha=0.5)
    x_line = np.linspace(0.0, 1.0, 300)
    if abs(w[1]) < 1e-8:
        x0 = mid[0]
        plt.axvline(x0)
    else:
        y_line = mid[1] + (w[0] / w[1]) * (mid[0] - x_line)
        mask = (y_line >= 0.0) & (y_line <= 1.0)
        plt.plot(x_line[mask], y_line[mask])
    plt.xlabel("mass_in_window")
    plt.ylabel("score")
    # add title with the q value and center_weight
    plt.title(f"Decision Boundary (q={q}, center_weight={center_weight})")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

# usage:
plot_decision_boundary(df, save_path=os.path.join(figuresDir, 'decision_boundary.png'))


In [ ]:
def compute_boundary_from_quadrants(df, q=0.3):
    x = df['mass_in_window'].values
    y = df['score'].values
    # Quantile thresholds instead of medians
    x_left  = np.quantile(x, q)
    x_right = np.quantile(x, 1 - q)
    y_low   = np.quantile(y, q)
    y_high  = np.quantile(y, 1 - q)
    # "Extreme" upper-left and lower-right
    mask_A = (x < x_left) & (y > y_high)
    mask_B = (x > x_right) & (y < y_low)
    S_A = np.column_stack([x[mask_A], y[mask_A]])
    S_B = np.column_stack([x[mask_B], y[mask_B]])
    if S_A.shape[0] == 0 or S_B.shape[0] == 0:
        raise ValueError("Not enough points in one of the anchor quadrants.")
    mu_A = S_A.mean(axis=0)
    mu_B = S_B.mean(axis=0)
    w   = mu_B - mu_A
    mid = 0.5 * (mu_A + mu_B)
    return w, mid


def compute_boundary_from_quadrants(df, q=0.3, center_weight=0.0):
    x = df['mass_in_window'].values
    y = df['score'].values
    x_left  = np.quantile(x, q)
    x_right = np.quantile(x, 1 - q)
    y_low   = np.quantile(y, q)
    y_high  = np.quantile(y, 1 - q)
    mask_A = (x < x_left) & (y > y_high)
    mask_B = (x > x_right) & (y < y_low)
    S_A = np.column_stack([x[mask_A], y[mask_A]])
    S_B = np.column_stack([x[mask_B], y[mask_B]])
    if S_A.shape[0] == 0 or S_B.shape[0] == 0:
        raise ValueError("Not enough points in one of the anchor quadrants.")
    mu_A = S_A.mean(axis=0)
    mu_B = S_B.mean(axis=0)
    w = mu_B - mu_A
    mid_data = 0.5 * (mu_A + mu_B)
    center   = np.array([0.5, 0.5])
    # center_weight in [0, 1]
    mid = (1 - center_weight) * mid_data + center_weight * center
    return w, mid


full_figs_dir = os.path.join(outDir, "figures_full")
os.makedirs(full_figs_dir, exist_ok=True)

for q in [0.1, 0.2, 0.3, .35, 0.4]:
    for center_weight in [0.0, 0.5, 1.0]:
        plot_decision_boundary(df, q=q, center_weight=center_weight, save_path=os.path.join(full_figs_dir, f'decision_boundary_q{q}_cw{center_weight}.png'))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# def compute_boundary_from_quadrants(df, q=0.3, center_weight=0.5):
#     x = df["mass_in_window"].values
#     y = df["score"].values
#     x_left = np.quantile(x, q)
#     x_right = np.quantile(x, 1 - q)
#     y_low = np.quantile(y, q)
#     y_high = np.quantile(y, 1 - q)
#     mask_A = (x < x_left) & (y > y_high)
#     mask_B = (x > x_right) & (y < y_low)
#     S_A = np.column_stack([x[mask_A], y[mask_A]])
#     S_B = np.column_stack([x[mask_B], y[mask_B]])
#     if S_A.shape[0] == 0 or S_B.shape[0] == 0:
#         raise ValueError("Not enough points in one of the anchor quadrants.")
#     mu_A = S_A.mean(axis=0)
#     mu_B = S_B.mean(axis=0)
#     w = mu_B - mu_A
#     mid_data = 0.5 * (mu_A + mu_B)
#     center = np.array([0.5, 0.5])
#     mid = (1 - center_weight) * mid_data + center_weight * center
#     return w, mid, S_A, S_B






def plot_boundary_with_band(df, q=0.3, center_weight=0.5, k_sigma=1.0, save_path=None):
    w, mid, S_A, S_B = compute_boundary_from_quadrants(
        df, q=q, center_weight=center_weight
    )
    w_norm = np.linalg.norm(w)
    if w_norm == 0:
        raise ValueError("Degenerate direction vector w has zero norm.")
    b = -np.dot(w, mid)
    # distances for scale: use anchor points, more "confident" regions
    anchors = np.vstack([S_A, S_B])
    d_anchors = (anchors @ w + b) / w_norm
    sigma = np.std(d_anchors)
    c = k_sigma * sigma
    x_vals = df["mass_in_window"].values
    y_vals = df["score"].values
    plt.figure(figsize=(6, 6))
    # Add log_size as the size of the points
    sizes = np.log1p(df["n_cells"].values) * 10
    # Remove the border of points
    plt.scatter(
        x_vals, y_vals, s=sizes, alpha=0.4, c=get_process_color(df), edgecolors="none"
    )
    x_line = np.linspace(0.0, 1.0, 400)
    if abs(w[1]) < 1e-8:
        # Error this out
        raise ValueError("Degenerate direction vector w has zero norm.")
        x0_center = -b / w[0]
        x0_up = -(b - c * w_norm) / w[0]
        x0_down = -(b + c * w_norm) / w[0]
        plt.axvline(x0_center, linewidth=1, linestyle="--")
        plt.axvline(x0_up, linestyle="dotted", linewidth=0.5, color="gray")
        plt.axvline(x0_down, linestyle="dotted", linewidth=0.5, color="gray")
    else:
        # central line: w0*x + w1*y + b = 0  ->  y = (-w0*x - b)/w1
        y_center = (-w[0] * x_line - b) / w[1]
        # band borders: w0*x + w1*y + b = ± c * ||w||
        y_up = (-w[0] * x_line - (b - c * w_norm)) / w[1]
        y_down = (-w[0] * x_line - (b + c * w_norm)) / w[1]
        mask_center = (y_center >= 0.0) & (y_center <= 1.0)
        mask_up = (y_up >= 0.0) & (y_up <= 1.0)
        mask_down = (y_down >= 0.0) & (y_down <= 1.0)
        plt.plot(
            x_line[mask_center], y_center[mask_center], linewidth=1, linestyle="--"
        )
        plt.plot(
            x_line[mask_up],
            y_up[mask_up],
            linestyle="dotted",
            linewidth=0.5,
            color="gray",
        )
        plt.plot(
            x_line[mask_down],
            y_down[mask_down],
            linestyle="dotted",
            linewidth=0.5,
            color="gray",
        )
    # Add text
    plt.grid(True)
    texts = []
    for i, row in df_label.iterrows():
        _ = texts.append(
            plt.text(
                row["mass_in_window"], row["score"], str(row["individual"]), fontsize=4
            )
        )
    _ = adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
    plt.xlabel("Mass in Window")
    plt.ylabel("ecDNA Score")
    plt.xlim(-0.1, 1.1)
    plt.ylim(-0.1, 1.1)
    # legend
    processes = df["process"].unique()
    colors = plt.cm.get_cmap("tab10", len(processes))
    for i, process in enumerate(processes):
        plt.scatter([], [], c=[colors(i)], label=process)
    # Put legend on the upper right
    # add a Title that has q and center_weight and k_sigma
    plt.title(f"Decision Boundary with Band (q={q}, cw={center_weight}, kσ={k_sigma})")
    plt.legend(loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()



In [ ]:

def compute_boundary_from_quadrants(df, q_A=0.2, q_B=0.3, center_weight=0.5):
    x = df["mass_in_window"].values
    y = df["score"].values
    x_left = np.quantile(x, q_A)
    x_right = np.quantile(x, 1 - q_B)
    y_low = np.quantile(y, q_B)
    y_high = np.quantile(y, 1 - q_A)
    mask_A = (x < x_left) & (y > y_high)
    mask_B = (x > x_right) & (y < y_low)
    S_A = np.column_stack([x[mask_A], y[mask_A]])
    S_B = np.column_stack([x[mask_B], y[mask_B]])
    if S_A.shape[0] == 0 or S_B.shape[0] == 0:
        raise ValueError("Not enough points in one of the anchor quadrants.")
    mu_A = S_A.mean(axis=0)
    mu_B = S_B.mean(axis=0)
    w = mu_B - mu_A
    mid_data = 0.5 * (mu_A + mu_B)
    center = np.array([0.5, 0.5])
    mid = (1 - center_weight) * mid_data + center_weight * center
    return w, mid, S_A, S_B

########### with q used to highlight quanta points
def plot_boundary_with_band(
    df,
    q_A=0.2,
    q_B=0.3,
    center_weight=0.5,
    k_sigma=1.0,
    df_label=None,
    save_path=None,
    highlight_quanta=True,
):
    w, mid, S_A, S_B = compute_boundary_from_quadrants(
        df, q_A=q_A, q_B=q_B, center_weight=center_weight
    )
    w_norm = np.linalg.norm(w)
    if w_norm == 0:
        raise ValueError("Degenerate direction vector w has zero norm.")
    b = -np.dot(w, mid)
    # distances for scale: use anchor points, more "confident" regions
    anchors = np.vstack([S_A, S_B])
    d_anchors = (anchors @ w + b) / w_norm
    sigma = np.std(d_anchors)
    c = k_sigma * sigma
    x_vals = df["mass_in_window"].values
    y_vals = df["score"].values
    plt.figure(figsize=(6, 6))
    # point sizes
    sizes = np.log1p(df["n_cells"].values) * 10
    # main scatter
    plt.scatter(
        x_vals,
        y_vals,
        s=sizes,
        alpha=0.4,
        c=get_process_color(df),
        edgecolors="none",
    )
    # highlight quanta points used for the boundary
    if highlight_quanta:
        # S_A and S_B are arrays of shape (n_quanta_A, 2) and (n_quanta_B, 2)
        quanta = np.vstack([S_A, S_B])
        plt.scatter(
            quanta[:, 0],
            quanta[:, 1],
            s=40,
            facecolors="none",
            edgecolors="k",
            linewidths=1.2,
            zorder=5,  # on top
            label="Quanta points",
        )
    # decision boundary and band
    x_line = np.linspace(0.0, 1.0, 400)
    if abs(w[1]) < 1e-8:
        # vertical boundary
        x0_center = -b / w[0]
        x0_up = -(b - c * w_norm) / w[0]
        x0_down = -(b + c * w_norm) / w[0]
        plt.axvline(x0_center, linewidth=1, linestyle="dashed")
        plt.axvline(x0_up, linestyle="dotted", linewidth=0.5, color="gray")
        plt.axvline(x0_down, linestyle="dotted", linewidth=0.5, color="gray")
    else:
        # central line: w0*x + w1*y + b = 0  ->  y = (-w0*x - b)/w1
        y_center = (-w[0] * x_line - b) / w[1]
        # band borders: w0*x + w1*y + b = ± c * ||w||
        y_up = (-w[0] * x_line - (b - c * w_norm)) / w[1]
        y_down = (-w[0] * x_line - (b + c * w_norm)) / w[1]
        mask_center = (y_center >= 0.0) & (y_center <= 1.0)
        mask_up = (y_up >= 0.0) & (y_up <= 1.0)
        mask_down = (y_down >= 0.0) & (y_down <= 1.0)
        plt.plot(
            x_line[mask_center], y_center[mask_center], linewidth=1, linestyle="dashed"
        )
        plt.plot(
            x_line[mask_up],
            y_up[mask_up],
            linestyle="dotted",
            linewidth=0.5,
            color="gray",
        )
        plt.plot(
            x_line[mask_down],
            y_down[mask_down],
            linestyle="dotted",
            linewidth=0.5,
            color="gray",
        )
    # labels with arrows
    plt.grid(True)
    texts = []
    # default to df if df_label is not provided
    if df_label is None:
        df_label = df
    for _, row in df_label.iterrows():
        t = plt.text(
            row["mass_in_window"],
            row["score"],
            str(row["individual"]),
            fontsize=4,
        )
        texts.append(t)
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
    plt.xlabel("Mass in Window")
    plt.ylabel("ecDNA Score")
    plt.xlim(-0.1, 1.1)
    plt.ylim(-0.1, 1.1)
    # legend for processes
    processes = df["process"].unique()
    colors = plt.cm.get_cmap("tab10", len(processes))
    for i, process in enumerate(processes):
        plt.scatter([], [], c=[colors(i)], label=process)
    plt.title(f"Decision Boundary with Band (q_A={q_A}, q_B={q_B}, cw={center_weight}, kσ={k_sigma})")
    plt.legend(loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()



In [ ]:
# Annotate the boundary zone
# TODO: make this more general
df_label = df[
    (df["mass_in_window"] > 0.4)
    & (df["mass_in_window"] < 0.8)
    & (df["score"] > 0.3)
    & (df["score"] < 0.6)
]

param_dict = {
    "config1": {"q_A": 0.2, "q_B": 0.3, "center_weight": 0.5, "k_sigma": 0.3},
    "config2": {"q_A": 0.05, "q_B": 0.3, "center_weight": 0.5, "k_sigma": 0.3},
    "config3": {"q_A": 0.01, "q_B": 0.3, "center_weight": 0.5, "k_sigma": 0.3},
    "config4": {"q_A": 0.1, "q_B": 0.3, "center_weight": 0.5, "k_sigma": 0.3},
    "config5": {"q_A": 0.15, "q_B": 0.3, "center_weight": 0.5, "k_sigma": 0.3},
}

for config_name, params in param_dict.items():
    try:
        plot_boundary_with_band(
            df,
            q_A=params["q_A"],
            q_B=params["q_B"],
            center_weight=params["center_weight"],
            k_sigma=params["k_sigma"],
            df_label=df_label,
            save_path=os.path.join(
                figuresDir,
                f"decision_boundary_with_band_{config_name}_qA{params['q_A']}_qB{params['q_B']}_cw_{params['center_weight']}_ks_{params['k_sigma']}.pdf",
            ),
        )
    except ValueError as e:
        print(f"Skipping {config_name} due to error: {e}")


In [ ]:

# Annotate the boundary zone
# TODO: make this more general
df_label = df[
    (df["mass_in_window"] > 0.4)
    & (df["mass_in_window"] < 0.8)
    & (df["score"] > 0.3)
    & (df["score"] < 0.6)
]

param_dict = {
    "config1": {"q": 0.35, "center_weight": 0.5, "k_sigma": 0.3},
    "config2": {"q": 0.05, "center_weight": 0.5, "k_sigma": 0.3},
    "config3": {"q": 0.1, "center_weight": 0.5, "k_sigma": 0.3},
    "config4": {"q": 0.06, "center_weight": 0.5, "k_sigma": 0.3},
}

for config_name, params in param_dict.items():
    plot_boundary_with_band(
        df,
        q=params["q"],
        center_weight=params["center_weight"],
        k_sigma=params["k_sigma"],
        df_label=df_label,
        save_path=os.path.join(
            figuresDir,
            f"decision_boundary_with_band_{config_name}_q{params['q']}_cw_{params['center_weight']}_ks_{params['k_sigma']}.pdf",
        ),
    )



In [ ]:


plot_boundary_with_band(
    df,
    q=0.35,
    center_weight=0.5,
    k_sigma=0.3,
    df_label=df_label,
    save_path=os.path.join(
        figuresDir, f"decision_boundary_with_band_q0.35_cw_0.5_ks_0.3.pdf"
    ),
)

plot_boundary_with_band(
    df,
    q=0.05,
    center_weight=0.5,
    k_sigma=0.3,
    save_path=os.path.join(figuresDir, "decision_boundary_with_band.pdf"),
)

plot_boundary_with_band(
    df,
    q=0.1,
    center_weight=0.5,
    k_sigma=0.3,
    save_path=os.path.join(figuresDir, "decision_boundary_with_band.pdf"),
)

#### Add n cells in decision boundary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def compute_boundary_from_quadrants_weighted(
    df, q=0.3, center_weight=0.5, weight_col="n_cells"
):
    x = df["mass_in_window"].values
    y = df["score"].values
    w_cells = df[weight_col].values.astype(float)
    x_left = np.quantile(x, q)
    x_right = np.quantile(x, 1 - q)
    y_low = np.quantile(y, q)
    y_high = np.quantile(y, 1 - q)
    mask_A = (x < x_left) & (y > y_high)
    mask_B = (x > x_right) & (y < y_low)
    if not (mask_A.any() and mask_B.any()):
        raise ValueError("Not enough points in one of the anchor quadrants.")
    x_A = x[mask_A]
    y_A = y[mask_A]
    w_A = w_cells[mask_A]
    x_B = x[mask_B]
    y_B = y[mask_B]
    w_B = w_cells[mask_B]
    mu_A_x = np.average(x_A, weights=w_A)
    mu_A_y = np.average(y_A, weights=w_A)
    mu_B_x = np.average(x_B, weights=w_B)
    mu_B_y = np.average(y_B, weights=w_B)
    mu_A = np.array([mu_A_x, mu_A_y])
    mu_B = np.array([mu_B_x, mu_B_y])
    w = mu_B - mu_A
    mid_data = 0.5 * (mu_A + mu_B)
    center = np.array([0.5, 0.5])
    mid = (1 - center_weight) * mid_data + center_weight * center
    # also return weighted anchors for band calculation
    S_A = np.column_stack([x_A, y_A])
    S_B = np.column_stack([x_B, y_B])
    return w, mid, S_A, S_B, w_A, w_B


def plot_boundary_with_band_weighted(
    df, q=0.3, center_weight=0.5, k_sigma=1.0, weight_col="n_cells", save_path=None
):
    w, mid, S_A, S_B, w_A, w_B = compute_boundary_from_quadrants_weighted(
        df, q=q, center_weight=center_weight, weight_col=weight_col
    )
    w_norm = np.linalg.norm(w)
    if w_norm == 0:
        raise ValueError("Degenerate direction vector w has zero norm.")
    b = -np.dot(w, mid)
    anchors = np.vstack([S_A, S_B])
    w_anchors = np.concatenate([w_A, w_B])
    d_anchors = (anchors @ w + b) / w_norm
    mean_d = np.average(d_anchors, weights=w_anchors)
    var_d = np.average((d_anchors - mean_d) ** 2, weights=w_anchors)
    sigma = np.sqrt(var_d)
    c = k_sigma * sigma
    x_vals = df["mass_in_window"].values
    y_vals = df["score"].values
    plt.figure()
    plt.scatter(x_vals, y_vals, s=10, alpha=0.4)
    x_line = np.linspace(0.0, 1.0, 400)
    if abs(w[1]) < 1e-8:
        x0_center = -b / w[0]
        x0_up = -(b - c * w_norm) / w[0]
        x0_down = -(b + c * w_norm) / w[0]
        plt.axvline(x0_center, linewidth=2)
        plt.axvline(x0_up, linestyle="--")
        plt.axvline(x0_down, linestyle="--")
    else:
        y_center = (-w[0] * x_line - b) / w[1]
        y_up = (-w[0] * x_line - (b - c * w_norm)) / w[1]
        y_down = (-w[0] * x_line - (b + c * w_norm)) / w[1]
        mask_center = (y_center >= 0.0) & (y_center <= 1.0)
        mask_up = (y_up >= 0.0) & (y_up <= 1.0)
        mask_down = (y_down >= 0.0) & (y_down <= 1.0)
        plt.plot(x_line[mask_center], y_center[mask_center], linewidth=2)
        plt.plot(x_line[mask_up], y_up[mask_up], linestyle="--")
        plt.plot(x_line[mask_down], y_down[mask_down], linestyle="--")
    plt.xlabel("mass_in_window")
    plt.ylabel("score")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()


# usage:
plot_boundary_with_band_weighted(
    df,
    q=0.3,
    center_weight=0.5,
    k_sigma=.5,
    weight_col="n_cells",
    save_path=os.path.join(figuresDir, "decision_boundary_with_band_weighted.png"),
)

## LDA based

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression


# # ---------------------------------------------------------
# # 1. Setup Mock Data (Replace this with your actual df)
# # ---------------------------------------------------------
# np.random.seed(42)
# n_points = 500
# df = pd.DataFrame({
#     'mass_in_window': np.concatenate([np.random.normal(0.2, 0.1, n_points//2), np.random.normal(0.8, 0.1, n_points//2)]),
#     'score': np.concatenate([np.random.normal(0.8, 0.1, n_points//2), np.random.normal(0.2, 0.1, n_points//2)]),
#     'n_cells': np.random.randint(10, 1000, n_points),
#     # Add some noise to make it realistic
# })
# # Clip to 0-1 range to match your domain
# df['mass_in_window'] = df['mass_in_window'].clip(0, 1)
# df['score'] = df['score'].clip(0, 1)


# ---------------------------------------------------------
# 2. The Principled LDA Function
# ---------------------------------------------------------
def fit_and_plot_lda_boundary(df, q=0.1, save_path=None, df_label=None, annotate_type=True):
    """
    Fits LDA on extreme 'anchor' points and projects the boundary 
    across the entire feature space.
    """
    x = df["mass_in_window"].values
    y = df["score"].values
    # --- A. Define Training Data (The Anchors) ---
    # We define labels purely based on your geometric quantiles
    x_left = np.quantile(x, q)
    x_right = np.quantile(x, 1 - q)
    y_low = np.quantile(y, q)
    y_high = np.quantile(y, 1 - q)
    # Class 0: Top-Left (Low Mass, High Score)
    mask_0 = (x < x_left) & (y > y_high)
    # Class 1: Bottom-Right (High Mass, Low Score)
    mask_1 = (x > x_right) & (y < y_low)
    # Combine these into a training set
    X_train = np.vstack([
        np.column_stack([x[mask_0], y[mask_0]]),
        np.column_stack([x[mask_1], y[mask_1]])
    ])
    # Create labels (0 for Group A, 1 for Group B)
    y_train = np.concatenate([np.zeros(mask_0.sum()), np.ones(mask_1.sum())])
    if len(np.unique(y_train)) < 2:
        raise ValueError("Not enough extreme points found to form two classes. Increase q.")
    # --- B. Train LDA ---
    # LDA finds the axis that maximizes separation relative to variance
    clf = LinearDiscriminantAnalysis(store_covariance=True)
    clf.fit(X_train, y_train)
    # --- C. Extract the Boundary Equation ---
    # Decision boundary is where w0*x + w1*y + b = 0
    w = clf.coef_[0]
    b = clf.intercept_[0]
    # --- D. Plotting ---
    plt.figure(figsize=(6, 6))
    # 1. Plot all data (gray/neutral)
    sizes = np.log1p(df["n_cells"].values) * 10
    # If annotate_type is true, add a scatter_plot with just the boundary underneat the points with the process colors
    if annotate_type:
        plt.scatter(
            x,
            y,
            s=sizes+.1,
            c=get_process_color(df),
            alpha=0.5,
            edgecolors="none",
        )
    plt.scatter(x, y, s=sizes, color='lightgray', alpha=0.5, label='Unclassified / Middle')
    # 2. Highlight the Training Anchors (to show what LDA learned from)
    plt.scatter(x[mask_0], y[mask_0], s=sizes[mask_0], color='teal', alpha=0.8, label='Training Class 0')
    plt.scatter(x[mask_1], y[mask_1], s=sizes[mask_1], color='orange', alpha=0.8, label='Training Class 1')
    # 3. Draw the Decision Boundary
    # Equation: y = -(w0 * x + b) / w1
    x_grid = np.linspace(0, 1, 100)
    decision_boundary = -(w[0] * x_grid + b) / w[1]
    # Print the decision boundary line (i.e. y = mx + c)
    print(f"Decision boundary line: y = {-w[0]/w[1]:.3f}x + {-b/w[1]:.3f}")
    # 4. Draw Probability Bands (The "Principled" equivalent of your geometric band)
    # LDA is probabilistic. We can calculate the lines for 95% confidence.
    # We solve for: decision_function(x) = log_odds
    # log(0.95/0.05) approx +/- 2.94
    # This shows where the model is 95% sure it belongs to one class or the other
    proba_logit = np.log(0.95 / 0.05) # ~2.94
    boundary_upper = -(w[0] * x_grid + b - proba_logit) / w[1] # 95% prob class 1
    boundary_lower = -(w[0] * x_grid + b + proba_logit) / w[1] # 95% prob class 0
    # Plot lines
    plt.plot(x_grid, decision_boundary, 'k--', linewidth=2, label='LDA Boundary (P=0.5)')
    plt.plot(x_grid, boundary_upper, 'k:', linewidth=1, alpha=0.6, label='95% Confidence')
    plt.plot(x_grid, boundary_lower, 'k:', linewidth=1, alpha=0.6)
    # print the equations of the upper and lower boundaries
    print(f"Upper boundary line: y = {-w[0]/w[1]:.3f}x + {-(b - proba_logit)/w[1]:.3f}")
    print(f"Lower boundary line: y = {-w[0]/w[1]:.3f}x + {-(b + proba_logit)/w[1]:.3f}")
    # Now plot a line using the following slope and intercept
    slope = 0.859
    intercept = -0.184
    y_line_fixed = slope * x_grid + intercept
    #plt.plot(x_grid, y_line_fixed, 'r-', linewidth=1.5, label='Fixed Boundary (for reference)')
    # Styling
    plt.xlim(-0.1, 1.1)
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Mass in Window")
    plt.ylabel("Score")
    # If df_label is provided, add annotations
    if df_label is not None:
        texts = []
        for _, row in df_label.iterrows():
            t = plt.text(
                row["mass_in_window"],
                row["score"],
                str(row["individual"]),
                fontsize=4,
            )
            texts.append(t)
        adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
    plt.legend(loc='upper right', fontsize='small')
    plt.title(f"LDA Boundary (Trained on q={q} extremes)")
    plt.grid(True, linestyle=':', alpha=0.6)
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

# ---------------------------------------------------------
# 3. Usage
# ---------------------------------------------------------
q = 0.05
fit_and_plot_lda_boundary(df, q=q, save_path=os.path.join(figuresDir, f"lda_decision_boundary_q{q}.pdf"), df_label=df_label, annotate_type=True)



for q in [0.05, 0.1, 0.15, 0.2]:
    fit_and_plot_lda_boundary(df, q=q, save_path=os.path.join(figuresDir, f"lda_decision_boundary_q{q}.png"), df_label=df_label)





In [ ]:
def fit_and_plot_lda_boundary(df, q=0.1, save_path=None, df_label=None, annotate_type=True):
    """
    Fits LDA on extreme 'anchor' points and projects the boundary 
    across the entire feature space.
    """
    x = df["mass_in_window"].values
    y = df["score"].values
    # --- A. Define Training Data (The Anchors) ---
    # We define labels purely based on your geometric quantiles
    x_left = np.quantile(x, q)
    x_right = np.quantile(x, 1 - q)
    y_low = np.quantile(y, q)
    y_high = np.quantile(y, 1 - q)
    # Class 0: Top-Left (Low Mass, High Score)
    mask_0 = (x < x_left) & (y > y_high)
    # Class 1: Bottom-Right (High Mass, Low Score)
    mask_1 = (x > x_right) & (y < y_low)
    # Combine these into a training set
    X_train = np.vstack([
        np.column_stack([x[mask_0], y[mask_0]]),
        np.column_stack([x[mask_1], y[mask_1]])
    ])
    # Create labels (0 for Group A, 1 for Group B)
    y_train = np.concatenate([np.zeros(mask_0.sum()), np.ones(mask_1.sum())])
    if len(np.unique(y_train)) < 2:
        raise ValueError("Not enough extreme points found to form two classes. Increase q.")
    # --- B. Train LDA ---
    # LDA finds the axis that maximizes separation relative to variance
    clf = LinearDiscriminantAnalysis(store_covariance=True)
    clf.fit(X_train, y_train)
    # --- C. Extract the Boundary Equation ---
    # Decision boundary is where w0*x + w1*y + b = 0
    w = clf.coef_[0]
    b = clf.intercept_[0]
    # --- D. Plotting ---
    plt.figure(figsize=(6, 6))
    # 1. Plot all data (gray/neutral)
    sizes = np.log1p(df["n_cells"].values) * 10
    # If annotate_type is true, add a scatter_plot with just the boundary underneat the points with the process colors
    if annotate_type:
        plt.scatter(
            x,
            y,
            s=sizes+.1,
            c=get_process_color(df),
            alpha=0.5,
            edgecolors="none",
        )
    plt.scatter(x, y, s=sizes, color='lightgray', alpha=0.5, label='Unclassified / Middle')
    # 2. Highlight the Training Anchors (to show what LDA learned from)
    plt.scatter(x[mask_0], y[mask_0], s=sizes[mask_0], color='teal', alpha=0.8, label='Training Class 0')
    plt.scatter(x[mask_1], y[mask_1], s=sizes[mask_1], color='orange', alpha=0.8, label='Training Class 1')
    # 3. Draw the Decision Boundary
    # Equation: y = -(w0 * x + b) / w1
    x_grid = np.linspace(0, 1, 100)
    decision_boundary = -(w[0] * x_grid + b) / w[1]
    # Print the decision boundary line (i.e. y = mx + c)
    print(f"Decision boundary line: y = {-w[0]/w[1]:.3f}x + {-b/w[1]:.3f}")
    # 4. Draw Probability Bands (The "Principled" equivalent of your geometric band)
    # LDA is probabilistic. We can calculate the lines for 95% confidence.
    # We solve for: decision_function(x) = log_odds
    # log(0.95/0.05) approx +/- 2.94
    # This shows where the model is 95% sure it belongs to one class or the other
    ########################################################
    ########################################################
    # proba_logit = np.log(0.95 / 0.05) # ~2.94
    # boundary_upper = -(w[0] * x_grid + b - proba_logit) / w[1] # 95% prob class 1
    # boundary_lower = -(w[0] * x_grid + b + proba_logit) / w[1] # 95% prob class 0
    # # Plot lines
    # plt.plot(x_grid, decision_boundary, 'k--', linewidth=2, label='LDA Boundary (P=0.5)')
    # plt.plot(x_grid, boundary_upper, 'k:', linewidth=1, alpha=0.6, label='95% Confidence')
    # plt.plot(x_grid, boundary_lower, 'k:', linewidth=1, alpha=0.6)
    # --- C2. Calibrate LDA score -> probability using 1D logistic regression ---
    # LDA score for training anchors
    score_train = X_train @ w + b
    score_train = score_train.reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs")
    cal.fit(score_train, y_train)
    def score_for_p(p):
        logit_p = np.log(p / (1.0 - p))
        return (logit_p - cal.intercept_[0]) / cal.coef_[0, 0]
    s_center = score_for_p(0.5)   # score where P=0.5
    s_hi = score_for_p(0.95)      # score where P=0.95
    s_lo = score_for_p(0.05)      # score where P=0.05
    # --- D. Plotting the calibrated boundaries ---
    x_grid = np.linspace(0, 1, 100)
    decision_boundary = -(w[0] * x_grid + b - s_center) / w[1]
    boundary_upper = -(w[0] * x_grid + b - s_hi) / w[1]
    boundary_lower = -(w[0] * x_grid + b - s_lo) / w[1]
    print(f"Decision boundary (P=0.5): y = {-w[0]/w[1]:.3f}x + {-(b - s_center)/w[1]:.3f}")
    print(f"Upper boundary (P=0.95): y = {-w[0]/w[1]:.3f}x + {-(b - s_hi)/w[1]:.3f}")
    print(f"Lower boundary (P=0.05): y = {-w[0]/w[1]:.3f}x + {-(b - s_lo)/w[1]:.3f}")
    plt.plot(x_grid, decision_boundary, "k--", linewidth=2, label="LDA boundary (P=0.5)")
    plt.plot(x_grid, boundary_upper, "k:", linewidth=1, alpha=0.6, label="P=0.95")
    plt.plot(x_grid, boundary_lower, "k:", linewidth=1, alpha=0.6, label="P=0.05")    
    ########################################################
    ########################################################
    #
    # print the equations of the upper and lower boundaries
    print(f"Upper boundary line: y = {-w[0]/w[1]:.3f}x + {-(b - proba_logit)/w[1]:.3f}")
    print(f"Lower boundary line: y = {-w[0]/w[1]:.3f}x + {-(b + proba_logit)/w[1]:.3f}")
    # Now plot a line using the following slope and intercept
    slope = 0.859
    intercept = -0.184
    y_line_fixed = slope * x_grid + intercept
    #plt.plot(x_grid, y_line_fixed, 'r-', linewidth=1.5, label='Fixed Boundary (for reference)')
    # Styling
    plt.xlim(-0.1, 1.1)
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Mass in Window")
    plt.ylabel("Score")
    # If df_label is provided, add annotations
    if df_label is not None:
        texts = []
        for _, row in df_label.iterrows():
            t = plt.text(
                row["mass_in_window"],
                row["score"],
                str(row["individual"]),
                fontsize=4,
            )
            texts.append(t)
        adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
    plt.legend(loc='upper right', fontsize='small')
    plt.title(f"LDA Boundary (Trained on q={q} extremes)")
    plt.grid(True, linestyle=':', alpha=0.6)
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

# ---------------------------------------------------------
# 3. Usage
# ---------------------------------------------------------
q = 0.05
fit_and_plot_lda_boundary(df, q=q, save_path=os.path.join(figuresDir, f"lda_decision_boundary_q{q}_calibrated.pdf"), df_label=df_label, annotate_type=True)

